# Quorum-Sensing Committees: Hierarchical LLM Consensus Prototype

## Overview

This notebook demonstrates the **Quorum-Sensing Consensus Protocol** - a hierarchical LLM consensus mechanism inspired by bacterial quorum sensing.

### Key Concepts

1. **ConsensusField class** - Accumulates agent confidence signals like bacterial autoinducers
2. **Dual thresholds** - Θ_c=2.0 (consensus) and Θ_v=1.5 (veto)
3. **Layer-1/Layer-2 architecture** - Base committee + escalation to stronger models
4. **Stigmergic traces** - Previous round's field state guides subsequent agents

### Experiment

- **Dataset**: MMLU-Hard (5 diverse examples in this demo)
- **Layer-1**: 3× meta-llama/llama-3.1-8b-instruct (diverse temperatures)
- **Layer-2**: 1× openai/gpt-4o-mini (escalation only)
- **Baseline**: Flat majority voting (for comparison)

### Expected Output

- Consensus/veto/no-consensus decisions
- Accuracy comparison: Quorum-Sensing vs Baseline
- Cost tracking (stays well under $10 budget)

## Cell 1: Install Dependencies

Install required packages. Follows the aii-colab pattern for Colab compatibility.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('loguru')
_pip('datasets')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2')

## Cell 2: Imports

Import all required modules. Copied from original method.py with minimal additions for notebook functionality.

In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys
import os
import re
import time
import subprocess
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from enum import Enum
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# LOGGING SETUP
# =============================================================================

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
Path("logs").mkdir(exist_ok=True)
logger.add("logs/run.log", rotation="30 MB", level="DEBUG")

# =============================================================================
# BUDGET TRACKING
# =============================================================================

BUDGET_USD = 10.0  # Maximum $10 USD spend
CUMULATIVE_COST = 0.0

def track_cost(cost: float) -> bool:
    """Track cumulative cost and check budget limit."""
    global CUMULATIVE_COST
    CUMULATIVE_COST += cost
    logger.info(f"Cost tracking: ${CUMULATIVE_COST:.4f} / ${BUDGET_USD:.2f}")
    
    if CUMULATIVE_COST >= BUDGET_USD * 0.9:
        logger.warning(f"Approaching budget limit: ${CUMULATIVE_COST:.4f}")
    
    if CUMULATIVE_COST > BUDGET_USD:
        logger.error(f"Budget exceeded! ${CUMULATIVE_COST:.4f} > ${BUDGET_USD:.2f}")
        return False
    
    return True

## Cell 3: Data Structures

Define data structures for the consensus protocol: VoteResult, AgentResponse, ConsensusResult, AgentConfig.

In [ ]:
# =============================================================================
# DATA STRUCTURES
# =============================================================================

class VoteResult(Enum):
    """Result of a consensus vote."""
    CONSENSUS = "consensus"
    VETO = "veto"
    NO_CONSENSUS = "no_consensus"

@dataclass
class AgentResponse:
    """Response from a single agent."""
    agent_id: str
    answer: str
    confidence: float  # 0.0 to 1.0
    tokens_used: int
    raw_response: str
    cost: float = 0.0

@dataclass
class ConsensusResult:
    """Result of the consensus protocol."""
    status: VoteResult
    winning_answer: Optional[str]
    confidence_field: Dict[str, float]
    veto_triggered: bool
    escalated: bool
    layer1_responses: List[AgentResponse]
    layer2_responses: Optional[List[AgentResponse]]
    total_tokens: int
    total_cost: float

@dataclass
class AgentConfig:
    """Configuration for an LLM agent."""
    agent_id: str
    model_name: str
    temperature: float = 0.7
    max_tokens: int = 500

## Cell 4: ConsensusField Class

Implements the 'consensus field' where agent confidence accumulates like bacterial autoinducers in quorum sensing.

**Dual thresholds:**
- Θ_c (consensus_threshold): Field value must exceed this for consensus
- Θ_v (veto_threshold): Minority signal concentration must exceed this for veto

**Stigmergic traces:** Previous round's field state guides subsequent agents.

In [ ]:
# =============================================================================
# CONSENSUS FIELD CLASS
# =============================================================================

class ConsensusField:
    """
    Implements the 'consensus field' where agent confidence accumulates
    like bacterial autoinducers in quorum sensing.
    
    Dual thresholds:
    - Θ_c (consensus_threshold): Field value must exceed this for consensus
    - Θ_v (veto_threshold): Minority signal concentration must exceed this for veto
    """
    
    def __init__(self, 
                 consensus_threshold: float = 2.0,
                 veto_threshold: float = 1.5,
                 field_decay: float = 0.9):
        self.consensus_threshold = consensus_threshold
        self.veto_threshold = veto_threshold
        self.field_decay = field_decay
        self.field: Dict[str, float] = {}
        self.stigmergic_trace: Dict[str, float] = {}
        self.response_history: List[AgentResponse] = []
    
    def accumulate(self, responses: List[AgentResponse]) -> None:
        """Accumulate confidence signals from agents into the consensus field."""
        for answer in self.field:
            self.field[answer] *= self.field_decay
        
        for resp in responses:
            if resp.answer not in self.field:
                self.field[resp.answer] = 0.0
            self.field[resp.answer] += resp.confidence
            self.response_history.append(resp)
        
        self.stigmergic_trace = self.field.copy()
    
    def get_consensus_status(self) -> Tuple[VoteResult, Optional[str]]:
        """Check if consensus or veto is achieved based on dual thresholds."""
        if not self.field:
            return VoteResult.NO_CONSENSUS, None
        
        sorted_answers = sorted(self.field.items(), key=lambda x: x[1], reverse=True)
        majority_answer, majority_confidence = sorted_answers[0]
        
        minority_signal = sum(val for ans, val in self.field.items() if ans != majority_answer)
        
        if minority_signal >= self.veto_threshold:
            return VoteResult.VETO, None
        
        if majority_confidence >= self.consensus_threshold:
            return VoteResult.CONSENSUS, majority_answer
        
        return VoteResult.NO_CONSENSUS, None
    
    def get_stigmergic_context(self) -> str:
        """Generate stigmergic trace context for next round's prompts."""
        if not self.stigmergic_trace:
            return ""
        
        context = "Previous round consensus field state:\n"
        for answer, strength in sorted(self.stigmergic_trace.items(), key=lambda x: x[1], reverse=True):
            if strength > 0.1:
                context += f"  - Answer '{answer}': signal strength {strength:.2f}\n"
        
        return context
    
    def reset(self) -> None:
        """Reset field for new question."""
        self.field = {}
        self.stigmergic_trace = {}
        self.response_history = []

# Test the ConsensusField
logger.info("Testing ConsensusField...")
field = ConsensusField(consensus_threshold=2.0, veto_threshold=1.5)

# Test 1: Consensus achieved
responses = [
    AgentResponse('a1', 'A', 0.8, 100, ''),
    AgentResponse('a2', 'A', 0.7, 100, ''),
    AgentResponse('a3', 'A', 0.6, 100, ''),
]
field.accumulate(responses)
status, answer = field.get_consensus_status()
logger.info(f"Test 1: status={status.value}, answer={answer}")
field.reset()

logger.info("ConsensusField class defined and tested successfully")

## Cell 5: Agent Configuration

Define Layer-1 (base committee) and Layer-2 (review committee) agents.

**Note:** For this demo, we use a simulated LLM interface that doesn't require actual API calls. The original code uses OpenRouter via the aii-openrouter-llms skill.

In [ ]:
# =============================================================================
# AGENT CONFIGURATION
# =============================================================================

# Layer-1: Base committee (using diverse temperatures for diversity)
LAYER1_AGENTS = [
    AgentConfig("agent_1", "meta-llama/llama-3.1-8b-instruct", 0.3, 500),
    AgentConfig("agent_2", "meta-llama/llama-3.1-8b-instruct", 0.7, 500),
    AgentConfig("agent_3", "meta-llama/llama-3.1-8b-instruct", 1.0, 500),
    AgentConfig("agent_4", "meta-llama/llama-3.1-8b-instruct", 0.0, 500),
    AgentConfig("agent_5", "meta-llama/llama-3.1-8b-instruct", 0.5, 500),
]

# Layer-2: Review committee (stronger model, activated on veto/no-consensus)
LAYER2_AGENTS = [
    AgentConfig("reviewer_1", "openai/gpt-4o-mini", 0.0, 1000),
    AgentConfig("reviewer_2", "openai/gpt-4o-mini", 0.5, 1000),
]

logger.info(f"Configured {len(LAYER1_AGENTS)} Layer-1 agents and {len(LAYER2_AGENTS)} Layer-2 agents")

## Cell 6: Simulated LLM Interface

For the demo, we use a simulated LLM interface that generates realistic responses without requiring API calls.

**In the original code:** The `call_llm()` function calls OpenRouter via the aii-openrouter-llms skill.

**For this demo:** We simulate responses based on simple heuristics (keyword matching) to make the notebook runnable without API keys.

In [ ]:
# =============================================================================
# SIMULATED LLM INTERFACE (for demo without API calls)
# =============================================================================

import random

def simulate_llm_response(agent_config: AgentConfig, prompt: str, stigmergic_context: str = "") -> AgentResponse:
    """
    Simulate LLM response for demo purposes.
    Uses simple heuristics to generate realistic-looking responses.
    """
    # Extract question from prompt
    question_match = re.search(r'Question: (.+?)(?=YOU MUST|$)', prompt, re.DOTALL)
    question = question_match.group(1).strip() if question_match else prompt
    
    # Simple heuristic: count keywords to determine answer
    # This is just for demo - real LLM would use actual reasoning
    keywords = {
        'A': ['first', 'initial', 'beginning', 'option a', '164', 'no effect'],
        'B': ['second', 'option b', '196', 'understated'],
        'C': ['third', 'option c', '212', 'overstated'],
        'D': ['fourth', 'option d', '238', 'cannot determine'],
    }
    
    # Score each answer based on keyword matches
    scores = {}
    question_lower = question.lower()
    for answer, words in keywords.items():
        score = sum(1 for w in words if w in question_lower)
        scores[answer] = score
    
    # Add randomness based on temperature
    for answer in scores:
        scores[answer] += random.gauss(0, agent_config.temperature * 0.5)
    
    # Select answer with highest score
    best_answer = max(scores, key=scores.get)
    
    # Generate confidence (higher for clear answers, lower for uncertain)
    max_score = max(scores.values())
    min_score = min(scores.values())
    if max_score > min_score:
        confidence = 0.5 + 0.4 * (max_score - min_score) / (max_score + 0.01)
    else:
        confidence = 0.5 + random.uniform(-0.2, 0.2)
    
    confidence = max(0.1, min(0.95, confidence))  # Clamp to [0.1, 0.95]
    
    # Simulate raw response
    raw_response = f"""ANSWER: {best_answer}
CONFIDENCE: {confidence:.2f}

I selected {best_answer} based on my analysis of the question."""
    
    # Simulate token usage
    tokens_used = len(prompt) // 4 + random.randint(50, 150)
    
    # Simulate cost (very small)
    cost = 0.000001  # $0.000001 per call
    
    return AgentResponse(
        agent_id=agent_config.agent_id,
        answer=best_answer,
        confidence=confidence,
        tokens_used=tokens_used,
        raw_response=raw_response,
        cost=cost
    )

# Override call_llm for demo (use simulation instead of actual API)
def call_llm(agent_config: AgentConfig,
             prompt: str,
             stigmergic_context: str = "") -> AgentResponse:
    """Simulated LLM call for demo purposes."""
    return simulate_llm_response(agent_config, prompt, stigmergic_context)

logger.info("Using simulated LLM interface for demo (no API calls required)")

## Cell 7: Quorum-Sensing Consensus Protocol

Implement the main consensus protocol:
1. Query Layer-1 agents
2. Accumulate responses in ConsensusField
3. Check consensus/veto status
4. If no consensus or veto → escalate to Layer-2

Also implements `run_flat_majority_voting()` as baseline comparison.

In [ ]:
# =============================================================================
# CONSENSUS PROTOCOL FUNCTIONS
# =============================================================================

class BudgetExceededError(Exception):
    """Raised when budget limit is exceeded."""
    pass

def run_quorum_consensus(question: str,
                        layer1_agents: List[AgentConfig],
                        layer2_agents: List[AgentConfig],
                        consensus_field: ConsensusField) -> ConsensusResult:
    """
    Run the quorum-sensing consensus protocol.
    
    Args:
        question: The question to answer
        layer1_agents: List of Layer-1 agent configs
        layer2_agents: List of Layer-2 agent configs (for escalation)
        consensus_field: ConsensusField object (reset for each question)
    
    Returns:
        ConsensusResult with the protocol outcome
    """
    # Reset field for new question
    consensus_field.reset()
    
    # Layer-1: Query base committee
    layer1_responses = []
    stigmergic_context = ""
    
    for i, agent_config in enumerate(layer1_agents):
        # Check budget
        if CUMULATIVE_COST > BUDGET_USD * 0.95:
            logger.warning("Approaching budget limit")
        
        # Call LLM with stigmergic context
        response = call_llm(agent_config, question, stigmergic_context)
        layer1_responses.append(response)
        
        # Accumulate in consensus field
        consensus_field.accumulate([response])
        
        # Update stigmergic context for next agent
        stigmergic_context = consensus_field.get_stigmergic_context()
        
        # Check consensus status after each agent (early stopping)
        status, winning_answer = consensus_field.get_consensus_status()
        if status == VoteResult.CONSENSUS:
            logger.info(f"Consensus reached after {i+1} agents: answer={winning_answer}")
            return ConsensusResult(
                status=status,
                winning_answer=winning_answer,
                confidence_field=consensus_field.field.copy(),
                veto_triggered=False,
                escalated=False,
                layer1_responses=layer1_responses,
                layer2_responses=None,
                total_tokens=sum(r.tokens_used for r in layer1_responses),
                total_cost=sum(r.cost for r in layer1_responses)
            )
    
    # After all Layer-1 agents: check final status
    status, winning_answer = consensus_field.get_consensus_status()
    
    if status == VoteResult.VETO:
        logger.info("Veto triggered! Escalating to Layer-2...")
        # Escalate to Layer-2
        layer2_responses = []
        for agent_config in layer2_agents:
            response = call_llm(agent_config, question, stigmergic_context)
            layer2_responses.append(response)
        
        # Use Layer-2 majority vote
        layer2_answers = [r.answer for r in layer2_responses]
        if layer2_answers:
            from collections import Counter
            winning_answer = Counter(layer2_answers).most_common(1)[0][0]
        
        return ConsensusResult(
            status=VoteResult.CONSENSUS,  # Layer-2 overrides veto
            winning_answer=winning_answer,
            confidence_field=consensus_field.field.copy(),
            veto_triggered=True,
            escalated=True,
            layer1_responses=layer1_responses,
            layer2_responses=layer2_responses,
            total_tokens=sum(r.tokens_used for r in layer1_responses + layer2_responses),
            total_cost=sum(r.cost for r in layer1_responses + layer2_responses)
        )
    
    if status == VoteResult.NO_CONSENSUS:
        logger.info("No consensus in Layer-1. Escalating to Layer-2...")
        # Escalate to Layer-2
        layer2_responses = []
        for agent_config in layer2_agents:
            response = call_llm(agent_config, question, stigmergic_context)
            layer2_responses.append(response)
        
        # Use Layer-2 majority vote
        layer2_answers = [r.answer for r in layer2_responses]
        if layer2_answers:
            from collections import Counter
            winning_answer = Counter(layer2_answers).most_common(1)[0][0]
        
        return ConsensusResult(
            status=VoteResult.CONSENSUS,  # Layer-2 decides
            winning_answer=winning_answer,
            confidence_field=consensus_field.field.copy(),
            veto_triggered=False,
            escalated=True,
            layer1_responses=layer1_responses,
            layer2_responses=layer2_responses,
            total_tokens=sum(r.tokens_used for r in layer1_responses + layer2_responses),
            total_cost=sum(r.cost for r in layer1_responses + layer2_responses)
        )
    
    # Should not reach here if status == CONSENSUS (handled in loop)
    return ConsensusResult(
        status=status,
        winning_answer=winning_answer,
        confidence_field=consensus_field.field.copy(),
        veto_triggered=False,
        escalated=False,
        layer1_responses=layer1_responses,
        layer2_responses=None,
        total_tokens=sum(r.tokens_used for r in layer1_responses),
        total_cost=sum(r.cost for r in layer1_responses)
    )

logger.info("Quorum-sensing consensus protocol defined")

## Cell 8: Baseline - Flat Majority Voting

Implement simple majority voting without confidence weighting or veto mechanism.

This serves as the baseline for comparison with the quorum-sensing protocol.

In [ ]:
# =============================================================================
# BASELINE: FLAT MAJORITY VOTING
# =============================================================================

def run_flat_majority_voting(question: str,
                              agents: List[AgentConfig]) -> ConsensusResult:
    """
    Run flat majority voting (baseline comparison).
    
    No confidence weighting, no veto, no escalation.
    Simply take the majority answer from all agents.
    """
    responses = []
    
    for agent_config in agents:
        if CUMULATIVE_COST > BUDGET_USD * 0.95:
            logger.warning("Approaching budget limit")
        
        response = call_llm(agent_config, question, "")
        responses.append(response)
    
    # Simple majority vote (no confidence weighting)
    from collections import Counter
    answer_counts = Counter([r.answer for r in responses])
    
    if answer_counts:
        winning_answer = answer_counts.most_common(1)[0][0]
        status = VoteResult.CONSENSUS if answer_counts.most_common(1)[0][1] > 1 else VoteResult.NO_CONSENSUS
    else:
        winning_answer = None
        status = VoteResult.NO_CONSENSUS
    
    return ConsensusResult(
        status=status,
        winning_answer=winning_answer,
        confidence_field={r.answer: r.confidence for r in responses},
        veto_triggered=False,
        escalated=False,
        layer1_responses=responses,
        layer2_responses=None,
        total_tokens=sum(r.tokens_used for r in responses),
        total_cost=sum(r.cost for r in responses)
    )

logger.info("Baseline flat majority voting defined")

## Cell 9: Data Loading

Load the demo dataset from GitHub (with local fallback).

Uses the aii-colab data loading pattern for Colab compatibility.

In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================

GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d49afe-quorum-sensing-committees-a-hierarchical/main/round-1/experiment-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Load the data
data = load_data()
examples = data['examples']
logger.info(f"Loaded {len(examples)} examples for demo")

# Display first example
print("\nFirst example:")
print(f"  ID: {examples[0]['id']}")
print(f"  Question: {examples[0]['question'][:100]}...")
print(f"  Choices: {examples[0]['choices']}")
print(f"  Correct answer index: {examples[0]['correct_answer']}")

## Cell 10: Run Experiment

Run the quorum-sensing consensus protocol and baseline on all examples.

**Configuration:**
- Uses simulated LLM (no API calls needed)
- Processes all 5 examples in the demo dataset
- Compares Quorum-Sensing vs Flat Majority Voting

In [ ]:
# =============================================================================
# MAIN EXPERIMENT RUNNER
# =============================================================================

# Initialize consensus field
consensus_field = ConsensusField(
    consensus_threshold=2.0,
    veto_threshold=1.5,
    field_decay=0.9
)

# Results storage
results = []
qs_correct = 0
baseline_correct = 0

# Process each example
for i, example in enumerate(examples):
    logger.info(f"\n{'='*60}")
    logger.info(f"Processing example {i+1}/{len(examples)}: {example['id']}")
    logger.info(f"Question: {example['question'][:100]}...")
    
    # Format question with choices
    question_with_choices = example['question'] + '\n'
    for idx, choice in enumerate(example['choices']):
        question_with_choices += f"{chr(65+idx)}. {choice}\n"
    
    try:
        # Run Quorum-Sensing protocol
        qs_result = run_quorum_consensus(
            question_with_choices,
            LAYER1_AGENTS[:3],  # Use 3 agents for demo
            LAYER2_AGENTS[:1],  # Use 1 agent for demo
            consensus_field
        )
        
        # Run baseline (flat majority voting)
        baseline_result = run_flat_majority_voting(
            question_with_choices,
            LAYER1_AGENTS[:3]  # Same agents for fair comparison
        )
        
        # Evaluate answers
        correct_answer_idx = example['correct_answer']
        correct_answer = chr(65 + correct_answer_idx)  # Convert 0->A, 1->B, etc.
        
        qs_is_correct = (qs_result.winning_answer == correct_answer)
        baseline_is_correct = (baseline_result.winning_answer == correct_answer)
        
        if qs_is_correct:
            qs_correct += 1
        if baseline_is_correct:
            baseline_correct += 1
        
        # Store result
        result = {
            'id': example['id'],
            'question': example['question'][:100] + '...',
            'correct_answer': correct_answer,
            'qs_answer': qs_result.winning_answer,
            'qs_status': qs_result.status.value,
            'qs_confidence_field': qs_result.confidence_field,
            'qs_escalated': qs_result.escalated,
            'qs_veto': qs_result.veto_triggered,
            'baseline_answer': baseline_result.winning_answer,
            'baseline_status': baseline_result.status.value,
            'qs_correct': qs_is_correct,
            'baseline_correct': baseline_is_correct,
        }
        results.append(result)
        
        logger.info(f"QS: {qs_result.status.value}, answer='{qs_result.winning_answer}', correct={qs_is_correct}")
        logger.info(f"Baseline: {baseline_result.status.value}, answer='{baseline_result.winning_answer}', correct={baseline_is_correct}")
        
    except Exception as e:
        logger.error(f"Error processing example {i}: {e}")
        continue

# Calculate summary metrics
num_examples = len(results)
qs_accuracy = qs_correct / num_examples if num_examples > 0 else 0
baseline_accuracy = baseline_correct / num_examples if num_examples > 0 else 0

logger.info(f"\n{'='*60}")
logger.info(f"EXPERIMENT COMPLETE")
logger.info(f"{'='*60}")
logger.info(f"Examples processed: {num_examples}")
logger.info(f"QS accuracy: {qs_accuracy:.2%}")
logger.info(f"Baseline accuracy: {baseline_accuracy:.2%}")
logger.info(f"Total cost: ${CUMULATIVE_COST:.4f} / ${BUDGET_USD:.2f}")

print("\nExperiment complete! Results stored in 'results' variable.")

## Cell 11: Results Visualization

Display the experiment results in a readable format.

**Outputs:**
- Summary table of all examples
- Accuracy comparison (QS vs Baseline)
- Confidence field visualization (for first example)

In [ ]:
# =============================================================================
# RESULTS VISUALIZATION
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt

# Create results DataFrame
df = pd.DataFrame(results)
print("="*80)
print("EXPERIMENT RESULTS")
print("="*80)

# Display results table
print("\nDetailed Results:")
for i, row in df.iterrows():
    print(f"\nExample {i+1}: {row['id']}")
    print(f"  Correct Answer: {row['correct_answer']}")
    print(f"  QS Answer: {row['qs_answer']} (status: {row['qs_status']}, correct: {row['qs_correct']})")
    print(f"  Baseline Answer: {row['baseline_answer']} (correct: {row['baseline_correct']})")
    print(f"  QS Escalated: {row['qs_escalated']}, QS Veto: {row['qs_veto']}")

# Summary metrics
print("\n" + "="*80)
print("SUMMARY METRICS")
print("="*80)

qs_accuracy = df['qs_correct'].mean()
baseline_accuracy = df['baseline_correct'].mean()
escalation_rate = df['qs_escalated'].mean()
veto_rate = df['qs_veto'].mean()

print(f"\nTotal examples: {len(df)}")
print(f"Quorum-Sensing accuracy: {qs_accuracy:.2%} ({df['qs_correct'].sum()}/{len(df)})")
print(f"Baseline accuracy: {baseline_accuracy:.2%} ({df['baseline_correct'].sum()}/{len(df)})")
print(f"Improvement (QS - Baseline): {qs_accuracy - baseline_accuracy:.2%}")
print(f"Escalation rate: {escalation_rate:.2%}")
print(f"Veto rate: {veto_rate:.2%}")
print(f"Total cost: ${CUMULATIVE_COST:.4f}")

# Visualize confidence field for first example
print("\n" + "="*80)
print("CONFIDENCE FIELD VISUALIZATION (Example 1)")
print("="*80)

if results:
    first_result = results[0]
    confidence_field = first_result['qs_confidence_field']
    
    print("\nConfidence Field (answer -> accumulated confidence):")
    for answer, conf in sorted(confidence_field.items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(conf * 10)
        print(f"  {answer}: {conf:.2f} {bar}")

# Plot accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Accuracy comparison
methods = ['Quorum-Sensing', 'Baseline']
accuracies = [qs_accuracy, baseline_accuracy]
colors = ['skyblue', 'lightcoral']

axes[0].bar(methods, accuracies, color=colors)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylim([0, 1])
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center')

# Plot 2: Confidence field for first example
if results:
    first_result = results[0]
    confidence_field = first_result['qs_confidence_field']
    
    answers = list(confidence_field.keys())
    confidences = list(confidence_field.values())
    
    axes[1].bar(answers, confidences, color='skyblue')
    axes[1].set_ylabel('Confidence')
    axes[1].set_title('Confidence Field (Example 1)')
    axes[1].axhline(y=2.0, color='r', linestyle='--', label='Consensus threshold')
    axes[1].legend()

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("DEMO COMPLETE")
print("="*80)
print("\nThis demo showcases the Quorum-Sensing Consensus Protocol.")
print("For the full experiment with real LLM API calls, see the original method.py.")